### 流式输出：`stream` 与 `astream`

`invoke()` 是「跑完整张图，只在最后返回一次最终状态」。流式接口把这个黑盒拆开：**每个节点执行完就吐一次数据**，LLM 甚至可以逐 token 吐。适合做进度条、分段展示、SSE 推前端。

LangGraph 的流式能力集中在两个方法上：

| 方法 | 形态 | 场景 |
| --- | --- | --- |
| `graph.stream(...)` | 同步生成器 | Notebook、脚本、CLI |
| `graph.astream(...)` | 异步生成器 | Web 服务里挂到 FastAPI/Starlette 的 SSE |

两者的参数与产出一一对应，只是同步/异步之分。所以「选哪种流式输出」这个问题，本质上是**选哪种 `stream_mode`**。

#### 本章示例用的图

先搭一张**不依赖大模型**的小图，专门用来把各种 `stream_mode` 的差别看清楚；讲到 `messages` 模式时再换成真实的 `ChatDeepSeek`。

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    topic: str
    draft: Annotated[str, operator.add]      # 累积：每次返回的都是增量
    steps: Annotated[list[str], operator.add]  # 累积：记录走过哪些节点


def outline(state: State):
    """列提纲：顺便用 stream_writer 往前端推一条进度。"""
    get_stream_writer()({"stage": "outline", "pct": 10})
    return {"draft": f"【提纲】{state['topic']}\n", "steps": ["outline"]}


def research(state: State):
    get_stream_writer()({"stage": "research", "pct": 40})
    return {"draft": "【资料】LangGraph 是有状态的多步工作流编排库。\n", "steps": ["research"]}


def write(state: State):
    get_stream_writer()({"stage": "write", "pct": 70})
    return {"draft": "【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。\n", "steps": ["write"]}


def polish(state: State):
    get_stream_writer()({"stage": "polish", "pct": 90})
    return {"draft": "【定稿】以上。\n", "steps": ["polish"]}


builder = StateGraph(State)
builder.add_node("outline", outline)
builder.add_node("research", research)
builder.add_node("write", write)
builder.add_node("polish", polish)

builder.add_edge(START, "outline")
builder.add_edge("outline", "research")     # 串行，保证 draft 的顺序
builder.add_edge("research", "write")
builder.add_edge("write", "polish")
builder.add_edge("polish", END)

graph = builder.compile()

INITIAL = {"topic": "LangGraph 是什么", "draft": "", "steps": []}

#### 先看一眼 `invoke` 的「黑盒」

跑完整张图，只在最后返回一个 state。中间发生了什么一概看不到——这就是需要流式的原因。

In [2]:
final = graph.invoke(dict(INITIAL))
print("steps =", final["steps"])
print("draft =\n" + final["draft"])

steps = ['outline', 'research', 'write', 'polish']
draft =
【提纲】LangGraph 是什么
【资料】LangGraph 是有状态的多步工作流编排库。
【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。
【定稿】以上。



### 1. `stream_mode="values"`：每一步的**完整状态**快照

每执行完一个 super-step，吐一次**归约之后的全量 state**。第一个 chunk 是入参本身。

- 优点：拿到的就是完整 state，不用自己拼；写文件、渲染进度条很方便。
- 缺点：状态大时重复传输多；且**并行节点在同一个 super-step 内的多次写入被合并成一次**（这里 `research` / `write` 是并行的，第 3 个 chunk 里两份内容都出现了），看不见单个节点干了什么。

In [3]:
for i, chunk in enumerate(graph.stream(dict(INITIAL), stream_mode="values")):
    print(f"--- values #{i} ---")
    print("steps:", chunk["steps"])
    print("draft:", repr(chunk["draft"]))

--- values #0 ---
steps: []
draft: ''
--- values #1 ---
steps: ['outline']
draft: '【提纲】LangGraph 是什么\n'
--- values #2 ---
steps: ['outline', 'research']
draft: '【提纲】LangGraph 是什么\n【资料】LangGraph 是有状态的多步工作流编排库。\n'
--- values #3 ---
steps: ['outline', 'research', 'write']
draft: '【提纲】LangGraph 是什么\n【资料】LangGraph 是有状态的多步工作流编排库。\n【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。\n'
--- values #4 ---
steps: ['outline', 'research', 'write', 'polish']
draft: '【提纲】LangGraph 是什么\n【资料】LangGraph 是有状态的多步工作流编排库。\n【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。\n【定稿】以上。\n'


### 2. `stream_mode="updates"`：**只吐增量**（默认模式）

每个节点返回什么就原样给什么，key 是节点名。而且**并行的节点各自单独成 chunk**，不合并——这是它和 `values` 最大的差别。

不给 `stream_mode` 时默认就是 `updates`。

In [4]:
# 默认就是 updates，下面两行等价
for chunk in graph.stream(dict(INITIAL)):
    (node_name, update), = chunk.items()
    print(f"[{node_name}] 新增 draft = {update['draft']!r}")

[outline] 新增 draft = '【提纲】LangGraph 是什么\n'
[research] 新增 draft = '【资料】LangGraph 是有状态的多步工作流编排库。\n'
[write] 新增 draft = '【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。\n'
[polish] 新增 draft = '【定稿】以上。\n'


In [5]:
# 显式写出来，方便对照
for chunk in graph.stream(dict(INITIAL), stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"[{node_name}] steps +{update['steps']}")

[outline] steps +['outline']
[research] steps +['research']
[write] steps +['write']
[polish] steps +['polish']


|  | `values` | `updates` |
| --- | --- | --- |
| 内容 | 每步的**全量 state** | 每个节点的**返回增量** |
| 并行节点 | 同一 super-step 合并成一个 chunk | 每个节点一个 chunk |
| 首个 chunk | 就是入参 | 无（还没执行） |
| 适合 | 存快照、渲染累计结果 | 观察「谁做了什么」、拼消息流 |

### 3. `stream_mode="custom"`：节点主动推进度

节点内部调 `get_stream_writer()(payload)` 写的内容，会被 `custom` 模式原样吐出，**不进 state、不影响图的状态流转**，纯旁路。这是做「正在检索资料…」「已生成 3/10」这类进度提示的标准做法。

注意：writer 传什么就出什么（可以是 dict、字符串），但它只在节点执行期间有效，且**必须在节点函数里调用**——在节点外调用会抛 `RuntimeError: Called get_config outside of a runnable context`。

In [6]:
print("invoke() 里这些进度全部被丢弃：", len(graph.invoke(dict(INITIAL))["steps"]), "步完成")

print("\ncustom 模式收到：")
for payload in graph.stream(dict(INITIAL), stream_mode="custom"):
    print("  ", payload)

invoke() 里这些进度全部被丢弃： 4 步完成

custom 模式收到：
   {'stage': 'outline', 'pct': 10}
   {'stage': 'research', 'pct': 40}
   {'stage': 'write', 'pct': 70}
   {'stage': 'polish', 'pct': 90}


`get_stream_writer()` 依赖 `contextvars` 传递，所以在 `async` 节点里同样能用（要求 Python >= 3.11）。

In [7]:
import asyncio


async def slow_write(state: State):
    """异步节点：每 200ms 推一条进度。"""
    writer = get_stream_writer()
    for i in range(3):
        writer({"stage": "writing", "pct": 50 + i * 10})
        await asyncio.sleep(0.2)
    return {"draft": "【异步正文】写完了。\n", "steps": ["slow_write"]}


async_builder = StateGraph(State)
async_builder.add_node("outline", outline)
async_builder.add_node("slow_write", slow_write)
async_builder.add_edge(START, "outline")
async_builder.add_edge("outline", "slow_write")
async_builder.add_edge("slow_write", END)
async_graph = async_builder.compile()


async def main():
    async for payload in async_graph.astream(dict(INITIAL), stream_mode="custom"):
        print("  ", payload)


await main()

   {'stage': 'outline', 'pct': 10}
   {'stage': 'writing', 'pct': 50}


   {'stage': 'writing', 'pct': 60}


   {'stage': 'writing', 'pct': 70}


### 4. `stream_mode="tasks"`：任务的开始与结束

每个节点（task）会吐**两次**：一次在执行前（含 `input`），一次在执行后（含 `result`、`error`、`interrupts`）。用来做「正在执行 X…」/「X 完成」的状态机，或者捕捉 `interrupts`——`chapter03/03-中断.ipynb` 里的人工中断就会在这里现形。

In [8]:
for chunk in graph.stream(dict(INITIAL), stream_mode="tasks"):
    if "input" in chunk:
        print(f"[{chunk['name']}] 开始")
    else:
        print(f"[{chunk['name']}] 结束，interrupts={chunk['interrupts']}，steps={chunk['result']['steps']}")

[outline] 开始
[outline] 结束，interrupts=[]，steps=['outline']
[research] 开始
[research] 结束，interrupts=[]，steps=['research']
[write] 开始
[write] 结束，interrupts=[]，steps=['write']
[polish] 开始
[polish] 结束，interrupts=[]，steps=['polish']


### 5. `stream_mode="checkpoints"`：快照流

每写一个 checkpoint 吐一次完整快照（含 `config` / `values` / `metadata` / `next` / `tasks`）。

**必须挂 checkpointer**，否则什么都不吐（`langgraph.checkpoint.memory.InMemorySaver` 够本地演示，生产用 Postgres，见 `chapter03/01-持久化.ipynb`）。这是做「实时调试 / 审计」时最省事的入口：不用等跑完，边跑边看状态怎么变。

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

cp_graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "demo-stream"}}

for chunk in cp_graph.stream(dict(INITIAL), config, stream_mode="checkpoints"):
    print(f"step={chunk['metadata'].get('step')} next={chunk['next']} steps={chunk['values']['steps']}")

step=-1 next=['__start__'] steps=[]
step=0 next=['outline'] steps=[]
step=1 next=['research'] steps=['outline']
step=2 next=['write'] steps=['outline', 'research']
step=3 next=['polish'] steps=['outline', 'research', 'write']
step=4 next=[] steps=['outline', 'research', 'write', 'polish']


### 6. `stream_mode="messages"`：LLM 的 token 级流

这是做「打字机效果」的模式。它把**节点里模型调用的流式 chunk** 直接漏出来，产出 `(chunk, metadata)` 二元组：

- `chunk` 是 `AIMessageChunk`（普通模型）或 `BaseMessageChunk`（tool 节点）。
- `metadata["langgraph_node"]` 告诉你这个 chunk 来自哪个节点。

⚠️ **坑：它发的是「节点产生的所有消息」，不只是模型输出。** 本仓库 `chapter01/06-预定义状态.ipynb` 就踩过：如果有节点往 state 里塞 `SystemMessage` / `HumanMessage`，这些也会以 `messages` 模式流出来。所以必须用 `langgraph_node` 过滤。

In [10]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import MessagesState

load_dotenv(override=True)

# deepseek-flash 支持 thinking，思考内容走 additional_kwargs["reasoning_content"]
llm = ChatDeepSeek(model="deepseek-flash", extra_body={"thinking": {"type": "enabled"}})


class ChatState(MessagesState):
    output: str


def build_prompt(state: ChatState):
    """这个节点自己造消息——它的输出也会被 messages 模式吐出来。"""
    return {
        "messages": [
            SystemMessage(content="你是个简洁的技术讲师，回答控制在两句话以内。"),
            HumanMessage(content=state["messages"][0].content),
        ]
    }


def chat(state: ChatState):
    resp = llm.invoke(state["messages"])
    return {"messages": [resp], "output": resp.content}


chat_builder = StateGraph(ChatState)
chat_builder.add_node("build_prompt", build_prompt)
chat_builder.add_node("chat", chat)
chat_builder.add_edge(START, "build_prompt")
chat_builder.add_edge("build_prompt", "chat")
chat_builder.add_edge("chat", END)
chat_graph = chat_builder.compile()

In [11]:
section = None   # 当前处于「思考」还是「回答」


def render(chunk, meta):
    """把 DeepSeek 的 reasoning + content 拼成带分隔的流式文本。"""
    global section
    if meta.get("langgraph_node") != "chat":   # ← 关键：滤掉 build_prompt 自己造的消息
        return
    reasoning = chunk.additional_kwargs.get("reasoning_content")
    if reasoning:
        if section != "thinking":
            print("\n=== 思考 ===")
            section = "thinking"
        print(reasoning, end="", flush=True)
    if chunk.content:
        if section != "answer":
            print("\n=== 回答 ===")
            section = "answer"
        print(chunk.content, end="", flush=True)


for chunk, meta in chat_graph.stream({"messages": [("user", "LangGraph 的核心价值是什么？")]}, stream_mode="messages"):
    render(chunk, meta)
print()


=== 思考 ===
We

 need

 answer

 in

 Chinese

...

维护

性

**

问题

。

### 7. 一次拿多种模式：`stream_mode=["updates", "custom"]`

传列表时产出变成 `(mode, chunk)` 二元组。这样可以在同一个循环里既渲染进度条、又更新状态，不用开两个 `stream()`（也就不会跑两遍图）。

注意多模式下的**交错顺序**：同一 super-step 里，先吐这个 step 里各节点发出的 `custom`，再吐它们的 `updates`。别假设模式是分组出现的。

In [12]:
for mode, chunk in graph.stream(dict(INITIAL), stream_mode=["custom", "updates"]):
    if mode == "custom":
        print(f"  进度 {chunk['pct']:>3}%  {chunk['stage']}")
    else:
        for node_name, update in chunk.items():
            print(f"  节点 {node_name} 返回 steps={update['steps']}")

  进度  10%  outline
  节点 outline 返回 steps=['outline']
  进度  40%  research
  节点 research 返回 steps=['research']
  进度  70%  write
  节点 write 返回 steps=['write']
  进度  90%  polish
  节点 polish 返回 steps=['polish']


### 8. `subgraphs=True`：穿透子图

默认情况下子图的流被包在父节点那一个 chunk 里。加上 `subgraphs=True` 后产出变成 `(namespace, chunk)`，namespace 是元组（空元组 `()` = 顶层图），子图节点的每一步都会单独冒出来。调试「图里嵌了图」时非常有用。

In [13]:
child_builder = StateGraph(State)
child_builder.add_node("write", write)
child_builder.add_node("polish", polish)
child_builder.add_edge(START, "write")
child_builder.add_edge("write", "polish")
child_builder.add_edge("polish", END)
child_graph = child_builder.compile()

parent_builder = StateGraph(State)
parent_builder.add_node("outline", outline)
parent_builder.add_node("child", child_graph)   # 编译后的图可以直接当节点用
parent_builder.add_edge(START, "outline")
parent_builder.add_edge("outline", "child")
parent_builder.add_edge("child", END)
parent_graph = parent_builder.compile()

In [14]:
print("--- 不加 subgraphs：子图被包成一个 chunk ---")
for chunk in parent_graph.stream(dict(INITIAL), stream_mode="updates"):
    print("  ", chunk)

print("\n--- subgraphs=True：按 namespace 拆开 ---")
for namespace, chunk in parent_graph.stream(dict(INITIAL), stream_mode="updates", subgraphs=True):
    who = "顶层" if not namespace else "子图 " + "/".join(namespace)
    print(f"   [{who}] {chunk}")

--- 不加 subgraphs：子图被包成一个 chunk ---
   {'outline': {'draft': '【提纲】LangGraph 是什么\n', 'steps': ['outline']}}
   {'child': {'topic': 'LangGraph 是什么', 'draft': '【提纲】LangGraph 是什么\n【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。\n【定稿】以上。\n', 'steps': ['outline', 'write', 'polish']}}

--- subgraphs=True：按 namespace 拆开 ---


   [顶层] {'outline': {'draft': '【提纲】LangGraph 是什么\n', 'steps': ['outline']}}
   [子图 child:86b9aafc-c3af-3dc9-b1f7-391c131c42e2] {'write': {'draft': '【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。\n', 'steps': ['write']}}
   [子图 child:86b9aafc-c3af-3dc9-b1f7-391c131c42e2] {'polish': {'draft': '【定稿】以上。\n', 'steps': ['polish']}}
   [顶层] {'child': {'topic': 'LangGraph 是什么', 'draft': '【提纲】LangGraph 是什么\n【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。\n【定稿】以上。\n', 'steps': ['outline', 'write', 'polish']}}


### 9. `astream`：异步流式

参数和产出与 `stream` 完全一致，只是变成 `async for`。**注意：想接 Web 服务，异步才是正确姿势**——FastAPI / Starlette 的 SSE handler 是 `async` 的，在里面跑同步 `stream()` 会堵住事件循环。

> Notebook 的 kernel 本身就跑在事件循环里，所以下面直接写 `await main()`。在普通 `.py` 脚本里则要 `asyncio.run(main())`；两种写法混用会报 `RuntimeError: asyncio.run() cannot be called from a running event loop`。

In [15]:
async def run_updates():
    print("--- astream + updates ---")
    async for chunk in graph.astream(dict(INITIAL), stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"  [async] {node_name}: {update['steps']}")


async def run_multi():
    print("\n--- astream + 多模式 ---")
    async for mode, chunk in graph.astream(dict(INITIAL), stream_mode=["custom", "updates"]):
        print(f"  [async] {mode}: {str(chunk)[:60]}")


async def run_parallel():
    """astream 的价值：多段并发流，互不阻塞。"""
    print("\n--- 两个协程各跑一遍 ---")

    async def one(tag):
        async for chunk in graph.astream(dict(INITIAL), stream_mode="updates"):
            for node_name in chunk:
                print(f"  [{tag}] {node_name}")
            await asyncio.sleep(0)   # 交还控制权，模拟真实 IO

    await asyncio.gather(one("A"), one("B"))


await run_updates()
await run_multi()
await run_parallel()

--- astream + updates ---
  [async] outline: ['outline']
  [async] research: ['research']
  [async] write: ['write']
  [async] polish: ['polish']

--- astream + 多模式 ---
  [async] custom: {'stage': 'outline', 'pct': 10}
  [async] updates: {'outline': {'draft': '【提纲】LangGraph 是什么\n', 'steps': ['outl
  [async] custom: {'stage': 'research', 'pct': 40}
  [async] updates: {'research': {'draft': '【资料】LangGraph 是有状态的多步工作流编排库。\n', 'st
  [async] custom: {'stage': 'write', 'pct': 70}
  [async] updates: {'write': {'draft': '【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 
  [async] custom: {'stage': 'polish', 'pct': 90}
  [async] updates: {'polish': {'draft': '【定稿】以上。\n', 'steps': ['polish']}}

--- 两个协程各跑一遍 ---


  [B] outline
  [A] outline
  [B] research
  [A] research
  [B] write
  [A] write
  [B] polish
  [A] polish


### 10. `astream_events`：拿底层 Runnable 的完整事件流

如果 `stream_mode` 那几种还不够——你想知道**模型 / 工具 / 链各自什么时候 start、什么时候吐 chunk、什么时候结束**——就用 `astream_events`。它是 LangChain 的统一事件协议，LangGraph 的图也是 `Runnable`，所以图上直接可用。

每个事件是个 dict，字段含义：

| 字段 | 说明 |
| --- | --- |
| `event` | `on_<runnable类型>_<start / stream / end>`，如 `on_chat_model_stream` |
| `name` | 发出事件的 Runnable 名字。图上就是节点名（顶层那条叫 `LangGraph`） |
| `run_id` / `parent_ids` | 嵌套调用链，可用来还原「谁的调用」 |
| `tags` | 图会打上 `graph:step:N`（第几个 super-step）、`graph:node:xxx` |
| `data` | 固定是 dict：`stream` 时为 `{"chunk": 增量}`，`start` 为 `{"input": ...}`，`end` 为 `{"output": 完整结果}` |

⚠️ **别把方法名写混**：`astream()` 也接受一个 `version` 参数，但那是 Pregel 自己的 stream 版本（`'v1'` / `'v2'`），和事件 schema 无关。事件流必须用 `astream_events(version=...)`。

In [16]:
async def survey_events():
    counts: dict[str, int] = {}
    # 用带模型的图，才能看到 on_chat_model_* 这类事件
    async for event in chat_graph.astream_events(
        {"messages": [("user", "用一句话介绍 LangGraph")]}, version="v2"
    ):
        counts[event["event"]] = counts.get(event["event"], 0) + 1
    for name, n in sorted(counts.items()):
        print(f"  {name:<22} {n}")


await survey_events()

  on_chain_end           3
  on_chain_start         3
  on_chain_stream        4
  on_chat_model_end      1
  on_chat_model_start    1
  on_chat_model_stream   184


In [17]:
async def walk_events():
    """不用模型的图，事件只有 chain 三类，结构最清楚。"""
    async for event in graph.astream_events(dict(INITIAL), version="v2"):
        tags = [t for t in event["tags"] if t.startswith("graph:")]
        print(f"  {event['event']:<20} name={event['name']:<10} tags={tags}")


await walk_events()

  on_chain_start       name=LangGraph  tags=[]
  on_chain_start       name=outline    tags=['graph:step:1']
  on_chain_stream      name=outline    tags=['graph:step:1']
  on_chain_end         name=outline    tags=['graph:step:1']
  on_chain_stream      name=LangGraph  tags=[]
  on_chain_start       name=research   tags=['graph:step:2']
  on_chain_stream      name=research   tags=['graph:step:2']
  on_chain_end         name=research   tags=['graph:step:2']
  on_chain_stream      name=LangGraph  tags=[]
  on_chain_start       name=write      tags=['graph:step:3']
  on_chain_stream      name=write      tags=['graph:step:3']
  on_chain_end         name=write      tags=['graph:step:3']
  on_chain_stream      name=LangGraph  tags=[]
  on_chain_start       name=polish     tags=['graph:step:4']
  on_chain_stream      name=polish     tags=['graph:step:4']
  on_chain_end         name=polish     tags=['graph:step:4']
  on_chain_stream      name=LangGraph  tags=[]
  on_chain_end         name=LangG

#### 用 `include_*` / `exclude_*` 过滤

不想被 `on_chain_stream` 淹没时，按名字或类型过滤更清爽。`include_types=["chat_model"]` 等价于只关心模型。

In [18]:
async def only_chain():
    print("--- include_names=['write', 'polish'] ---")
    async for event in graph.astream_events(
        dict(INITIAL), version="v2", include_names=["write", "polish"]
    ):
        print(f"  {event['event']:<20} {event['name']}")

    print("\n--- include_types=['chat_model']（只看模型事件）---")
    shown = 0
    async for event in chat_graph.astream_events(
        {"messages": [("user", "用一句话介绍 LangGraph")]},
        version="v2",
        include_types=["chat_model"],
    ):
        if event["event"] == "on_chat_model_stream":
            text = event["data"]["chunk"].content
            if text and shown < 8:      # 首个 chunk 常是空串（DeepSeek 会先吐 reasoning）
                shown += 1
                print("  ", repr(text))
        elif event["event"] == "on_chat_model_start":
            print("   on_chat_model_start：只看得到 input，模型还没开始吐字")
        elif event["event"] == "on_chat_model_end":
            print("   on_chat_model_end：output 是完整消息")


await only_chain()

--- include_names=['write', 'polish'] ---
  on_chain_start       write
  on_chain_stream      write
  on_chain_end         write
  on_chain_start       polish
  on_chain_stream      polish
  on_chain_end         polish

--- include_types=['chat_model']（只看模型事件）---
   on_chat_model_start：只看得到 input，模型还没开始吐字


   'Lang'
   'Graph'
   ' '


   '是'
   ' Lang'
   'Chain'
   ' '
   '推出的'
   on_chat_model_end：output 是完整消息


#### 自定义事件：`adispatch_custom_event`

`stream_mode="custom"` 走 `get_stream_writer()`，而 `astream_events` 走 `adispatch_custom_event(name, data)`，产出 `on_custom_event` 事件（带一个 `name` 字段，便于前端分流）。两者**不能互相替代**：writer 的内容不会出现在事件流里，`dispatch_custom_event` 的内容也不会出现在 `custom` 模式里。

In [19]:
from langchain_core.callbacks import adispatch_custom_event


async def chat_with_todo(state: ChatState):
    """模型回答的同时，往外抛一份「待办清单」。"""
    await adispatch_custom_event("todo_list", {"todos": ["回顾上一轮", "组织答案"]})
    resp = await llm.ainvoke(state["messages"])
    return {"messages": [resp], "output": resp.content}


todo_builder = StateGraph(ChatState)
todo_builder.add_node("chat_with_todo", chat_with_todo)
todo_builder.add_edge(START, "chat_with_todo")
todo_builder.add_edge("chat_with_todo", END)
todo_graph = todo_builder.compile()


async def show_custom_event():
    async for event in todo_graph.astream_events(
        {"messages": [("user", "LangGraph 的核心价值是什么？")]}, version="v2"
    ):
        if event["event"] == "on_custom_event":
            print(f"  自定义事件 name={event['name']} data={event['data']}\n")
        elif event["event"] == "on_chat_model_stream":
            # stream 事件的 data 是 {"chunk": ...}；end 事件则是 {"output": ...}
            # 思考内容走 reasoning_content，正文走 content；这里只看正文，免得刷屏
            text = event["data"]["chunk"].content
            if text:
                print(text, end="", flush=True)
        elif event["event"] == "on_chat_model_end":
            print("\n  ↑ 模型结束")


await show_custom_event()

  自定义事件 name=todo_list data={'todos': ['回顾上一轮', '组织答案']}



Lang

Graph

的核心

价值

...

生产

化

。

**


  ↑ 模型结束


### 11. `version="v3"`：投影式流协议（beta）

`version="v3"` 不再吐「一条条事件 dict」，而是返回一个**流对象**（`AsyncGraphRunStream`），上面挂着若干**投影**（projection）：想看哪路就迭代哪路。设计上是**调用方驱动**——你的循环本身就是泵，不额外起后台线程。

⚠️ 四点注意：
1. **必须先 `await` 拿到流对象**。`async for ev in graph.astream_events(..., version="v3")` 会报 `TypeError: 'async for' requires an object with __aiter__`，因为它返回的是 coroutine。
2. 目前是 `@beta`，运行时会打 `LangChainBetaWarning`，API 可能变。
3. **不接受 `stream_mode` / `subgraphs`**：传了直接 `TypeError`（v3 自己决定用哪些投影，且子图命名空间恒定打开）。
4. **投影是单消费者**：同一个投影迭代两次会报错，需要扇出用 `projection.tee(n)`。
5. **多路投影必须并发消费**：迭代任一投影就是在驱动图执行（它就是泵）。先迭代 `updates` 跑到底，再迭代 `custom` 只会拿到空列表——要用 `asyncio.gather` 同时 drain。

默认注册四个原生投影：`run.values`、`run.messages`、`run.lifecycle`（子图生命周期）、`run.subgraphs`。想拿 `updates` / `custom` 得显式传 `transformers`。

`run.values` 这一路**直接给 state dict**，不再需要从协议事件里 `ev["params"]["data"]` 拆出来——这正是「投影式」的好处。

In [20]:
async def v3_values():
    print("--- v3: run.values 直接给 state，不用拆 event['params'] ---")
    run = await graph.astream_events(dict(INITIAL), version="v3")
    print("  投影列表 =", list(run.extensions))

    async for state in run.values:      # 直接是 state dict，不是协议事件
        print("  steps =", state["steps"])

    final = await run.output()          # 把图跑到底并返回最终 state
    print("  output() ==", final["steps"])


await v3_values()

--- v3: run.values 直接给 state，不用拆 event['params'] ---
  投影列表 = ['values', 'messages', 'lifecycle', 'subgraphs']
  steps = []
  steps = ['outline']
  steps = ['outline', 'research']
  steps = ['outline', 'research', 'write']
  steps = ['outline', 'research', 'write', 'polish']
  output() == ['outline', 'research', 'write', 'polish']


/var/folders/c3/6hjnnz31245_jz48xqwyjbyr0000gn/T/ipykernel_61211/2536002600.py:3: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  run = await graph.astream_events(dict(INITIAL), version="v3")
/Users/lijixu/PycharmProjects/langgraph_demo/.venv/lib/python3.13/site-packages/langgraph/pregel/main.py:3612: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return AsyncGraphRunStream(graph_aiter, mux)


In [21]:
from langgraph.stream.transformers import CustomTransformer, UpdatesTransformer


async def drain(channel):
    return [item async for item in channel]


async def v3_updates():
    print("\n--- v3: 用 transformers 打开 updates / custom 投影 ---")
    run = await graph.astream_events(
        dict(INITIAL), version="v3", transformers=[UpdatesTransformer, CustomTransformer]
    )
    print("  投影列表 =", list(run.extensions))

    # 两个投影要**并发**消费：任一投影被迭代就是泵，谁先跑完谁就把图跑到底，
    # 之后再迭代另一个投影只会拿到空（顺序迭代的实测结果）。
    updates, custom = await asyncio.gather(
        drain(run.extensions["updates"]),
        drain(run.extensions["custom"]),
    )
    for chunk in updates:
        print("  updates:", chunk)
    for payload in custom:
        print("  custom:", payload)


await v3_updates()


--- v3: 用 transformers 打开 updates / custom 投影 ---
  投影列表 = ['values', 'messages', 'lifecycle', 'subgraphs', 'updates', 'custom']
  updates: {'outline': {'draft': '【提纲】LangGraph 是什么\n', 'steps': ['outline']}}
  updates: {'research': {'draft': '【资料】LangGraph 是有状态的多步工作流编排库。\n', 'steps': ['research']}}
  updates: {'write': {'draft': '【正文】它把 AI 应用拆成「节点 + 边」，并用 checkpointer 支持恢复。\n', 'steps': ['write']}}
  updates: {'polish': {'draft': '【定稿】以上。\n', 'steps': ['polish']}}
  custom: {'stage': 'outline', 'pct': 10}
  custom: {'stage': 'research', 'pct': 40}
  custom: {'stage': 'write', 'pct': 70}
  custom: {'stage': 'polish', 'pct': 90}


In [22]:
async def v3_messages():
    print("\n--- v3: run.messages（content block 协议，事件是扁平 dict）---")
    run = await chat_graph.astream_events(
        {"messages": [("user", "LangGraph 的核心价值是什么？")]}, version="v3"
    )
    async for model_stream in run.messages:      # 每个模型调用一个流对象
        async for ev in model_stream:
            # v3 的模型流是 content block 协议，事件是扁平 dict：
            # message-start / content-block-start / content-block-delta /
            # content-block-stop / message-stop
            if ev["event"] != "content-block-delta":
                continue
            delta = ev["delta"]
            text = delta.get("reasoning") or delta.get("content") or ""
            if text:
                kind = "reasoning" if delta.get("type") == "reasoning-delta" else "content"
                print(f"  block#{ev['index']} {kind}: {text!r}")
        break                                     # 只看第一次模型调用
    await run.output()


await v3_messages()


--- v3: run.messages（content block 协议，事件是扁平 dict）---


### 该用哪个？一张对照表

| 你想做的事 | 选择 |
| --- | --- |
| 观察每个节点返回了什么 | `stream_mode="updates"`（默认） |
| 每步存一次完整状态 / 渲染累计结果 | `stream_mode="values"` |
| 「正在检索…」这类进度提示 | 节点里 `get_stream_writer()` + `stream_mode="custom"` |
| 打字机效果 + 思考内容 | `stream_mode="messages"`（务必按 `langgraph_node` 过滤） |
| 边跑边看状态变化做调试 | 挂 checkpointer + `stream_mode="checkpoints"` |
| 捕捉中断 / 显示每个 task 的起止 | `stream_mode="tasks"` |
| 同时要多种视图 | `stream_mode=["updates", "custom"]`，解包 `(mode, chunk)` |
| 调试嵌套子图 | 加 `subgraphs=True`，解包 `(namespace, chunk)` |
| 想知道模型/工具/链的完整生命周期 | `astream_events(version="v2")` |
| 新项目想试投影式流 | `astream_events(version="v3")`（beta，需先 `await`，不支持 `stream_mode`） |
| Web 服务 / 需要并发 | 换成 `astream` / `astream_events` |

### 常见坑

1. **默认模式是 `updates`**，不是 `values`。想要全量快照必须显式写 `stream_mode="values"`。
2. **`messages` 模式会把节点自造的消息也发出来**（`SystemMessage` / `HumanMessage`），必须用 `metadata["langgraph_node"]` 过滤，否则会打印出你不想要的内容。
3. **`updates` 下并行节点各自成 chunk，`values` 下会合并**。所以「chunk 数 = 步数」这个直觉只在无并行时成立——super-step 才是真正的步（`recursion_limit` 折算的也是 super-step）。
4. **`get_stream_writer()` 只能在节点/entrypoint 内部调用**，在外面调会抛 `RuntimeError`；且 Python < 3.11 下异步调用失效（`contextvars` 限制）。
5. **`custom` 与 `adispatch_custom_event` 互不相通**：前者进 `custom` 模式，后者进 `astream_events` 的 `on_custom_event`。想两边都要就写两遍。
6. **`stream_mode="checkpoints"` 不挂 checkpointer 就静默返回空**，不是报错——很容易以为程序坏了。
7. **v3 必须先 `await`**，而且**不接受 `stream_mode` / `subgraphs`**：`async for ev in graph.astream_events(..., version="v3")` 报 `TypeError`（返回的是 coroutine）；`stream_mode="updates"` 报 `TypeError: stream_events(version='v3') do not accept stream_mode`。要 `updates` 得传 `transformers=[UpdatesTransformer]`。
8. **v3 多路投影要并发 drain**。迭代哪个投影哪个就是泵，顺序迭代会让后面的投影拿到空。
9. **流式下不要复用同一个生成器**。`graph.stream()` 每次调用都是一次新的图执行；想复用结果就 `list(...)` 存下来，或者挂 checkpointer 后续跑。

### 小结

- `stream` / `astream` 是同一套语法的同步与异步版本，差别只在 `stream_mode`。
- `updates` 是默认视图（增量、并行不合并），`values` 是全量快照（合并）。
- `custom` + `get_stream_writer()` 是节点向外推旁路数据的唯一方式，不污染 state。
- `messages` 是唯一能拿到 token 级流的模式，代价是要自己过滤节点。
- `astream_events` 把视野扩到整个 Runnable 树：v2 给事件 dict，v3（beta）给投影对象（`run.values` / `run.messages` / `run.extensions[...]`）。